# Seguridad de la API GraphQL

Endurecemos el esquema GraphQL del proyecto (`tp_mlops2.graphql_api.schema`).

**Riesgo:** por defecto, GraphQL expone **introspección** (un atacante puede pedir
`{ __schema { types { name } } }` y mapear el esquema completo sin credenciales), y
no limita la **profundidad** de una query — una consulta anidada hasta el linaje completo
en Neo4j es costosa de resolver, y nada impide pedirla miles de veces (DoS).

**Controles que agregamos:** desactivar introspección en producción, limitar la
profundidad de las queries, y autorización por campo (el campo `lineage`, que dispara
una consulta a Neo4j, solo para roles autorizados).

In [1]:
import strawberry
from strawberry.extensions import AddValidationRules, QueryDepthLimiter
from graphql import NoSchemaIntrospectionCustomRule

from tp_mlops2.graphql_api.schema import get_model

modelo_real = get_model()
print(f"Modelo real: {modelo_real.name} v{modelo_real.version}")
print(f"MAE: {modelo_real.metrics.mae_mw:.1f} MW | MAPE: {modelo_real.metrics.mape:.2%}")

2026/10/10 19:07:33 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Modelo real: random_forest_demanda v8
MAE: 1736.8 MW | MAPE: 6.00%


## 1. Estado actual: introspección abierta, sin límite de profundidad

El esquema real del proyecto (`strawberry.Schema(query=Query)`, sin extensiones) permite
que cualquiera mapee el esquema completo con una sola query.

In [2]:
from tp_mlops2.graphql_api.schema import Query as QueryReal

schema_v = strawberry.Schema(query=QueryReal)  # tal cual está en producción: sin límites

r = schema_v.execute_sync("{ __schema { types { name } } }")
tipos_expuestos = [t["name"] for t in r.data["__schema"]["types"]]
print(f"Tipos expuestos por introspección sin ningún control: {len(tipos_expuestos)}")
print(tipos_expuestos[:8], "...")

Tipos expuestos por introspección sin ningún control: 15
['Query', 'Model', 'String', 'Metrics', 'Float', 'LineageNode', 'Boolean', '__Schema'] ...


## 2. Sin introspección, límite de profundidad y autorización por campo

Reconstruimos el `Query` real, pero el campo `lineage` (el que dispara la consulta a
Neo4j) ahora exige que el rol del que llama esté en el contexto, y el esquema se sirve
sin introspección y con tope de profundidad.

In [3]:
from tp_mlops2.graphql_api.schema import LineageNode, Metrics, Model


@strawberry.type
class QuerySegura:
    @strawberry.field
    def model(self, info: strawberry.Info, name: str) -> Model:
        return modelo_real

    @strawberry.field
    def lineage(self, info: strawberry.Info) -> list[LineageNode]:
        # autorización por campo: el linaje completo (consulta a Neo4j) es más
        # costoso y más sensible que las métricas — solo para roles autorizados
        rol = info.context.get("rol")
        if rol not in ("analista", "admin"):
            raise Exception("no autorizado: el campo lineage requiere rol analista o admin")
        return modelo_real.lineage()


schema = strawberry.Schema(
    query=QuerySegura,
    extensions=[
        lambda: AddValidationRules([NoSchemaIntrospectionCustomRule]),
        lambda: QueryDepthLimiter(max_depth=4),
    ],
)

# introspección: ahora bloqueada
r_introspeccion = schema.execute_sync("{ __schema { types { name } } }")
print("introspección bloqueada:", bool(r_introspeccion.errors), "->", r_introspeccion.errors)

# query de profundidad razonable, con rol autorizado: pasa
r_con_rol = schema.execute_sync(
    '{ model(name: "random_forest_demanda") { name version metrics { maeMw mape } } }'
)
print("\nmodel() con query simple:", r_con_rol.errors or r_con_rol.data)

# lineage sin rol en el contexto: rechazado
r_sin_rol = schema.execute_sync("{ lineage { name kind } }", context_value={})
print("\nlineage sin rol:", r_sin_rol.errors)

# lineage con rol autorizado: pasa y trae el linaje real desde Neo4j
r_lineage_ok = schema.execute_sync("{ lineage { name kind } }", context_value={"rol": "analista"})
if r_lineage_ok.errors:
    print("\nlineage con rol analista: error ->", r_lineage_ok.errors)
else:
    nodos = r_lineage_ok.data["lineage"]
    print(f"\nlineage con rol analista: {len(nodos)} nodos reales desde Neo4j")

GraphQL introspection has been disabled, but the requested query contained the field '__schema'.

GraphQL request:1:3
1 | { __schema { types { name } } }
  |   ^


GraphQL introspection has been disabled, but the requested query contained the field 'types'.

GraphQL request:1:14
1 | { __schema { types { name } } }
  |              ^


no autorizado: el campo lineage requiere rol analista o admin

GraphQL request:1:3
1 | { lineage { name kind } }
  |   ^
Traceback (most recent call last):
  File "D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\graphql\execution\execute.py", line 562, in execute_field
    result = resolve_fn(source, info, **args)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\strawberry\schema\schema_converter.py", line 1055, in _resolver
    return _get_result_with_extensions(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\strawberry\schema\schema_converter.py", line 1041, in extension_resolver
    return reduce(
           ^^^^^^^
  File "D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de

introspección bloqueada: True -> [GraphQLError("GraphQL introspection has been disabled, but the requested query contained the field '__schema'.", locations=[SourceLocation(line=1, column=3)]), GraphQLError("GraphQL introspection has been disabled, but the requested query contained the field 'types'.", locations=[SourceLocation(line=1, column=14)])]

model() con query simple: {'model': {'name': 'random_forest_demanda', 'version': '8', 'metrics': {'maeMw': 1736.815230933058, 'mape': 0.05999729717375087}}}

lineage sin rol: [GraphQLError('no autorizado: el campo lineage requiere rol analista o admin', locations=[SourceLocation(line=1, column=3)], path=['lineage'])]

lineage con rol analista: 18 nodos reales desde Neo4j


## 3. Límite de profundidad en acción

Una query anidada varias veces (simulando un intento de abusar del grafo de linaje
pidiendo `lineage` repetidamente dentro de fragmentos) se corta antes de ejecutarse.

In [4]:
query_profunda = """
{
  model(name: "random_forest_demanda") {
    name
    metrics {
      maeMw
    }
  }
}
"""
# con max_depth=4 esta query (profundidad 3) pasa; subimos el límite a 1 para
# demostrar el corte sin tener que escribir una query artificialmente anidada.
schema_estricto = strawberry.Schema(
    query=QuerySegura,
    extensions=[
        lambda: AddValidationRules([NoSchemaIntrospectionCustomRule]),
        lambda: QueryDepthLimiter(max_depth=1),
    ],
)
r_profunda = schema_estricto.execute_sync(query_profunda)
print("profundidad 3 contra límite de 1:", r_profunda.errors)

'anonymous' exceeds maximum operation depth of 1

GraphQL request:6:7
5 |     metrics {
6 |       maeMw
  |       ^
7 |     }


profundidad 3 contra límite de 1: [GraphQLError("'anonymous' exceeds maximum operation depth of 1", locations=[SourceLocation(line=6, column=7)])]


## 4. Reflexión

**¿Por qué apagar la introspección en producción reduce el riesgo?**
La introspección le da a un atacante el mapa completo del esquema (tipos, campos,
relaciones) sin necesidad de adivinar nada — es el equivalente a publicar la
documentación interna de la API. Para este proyecto en particular, expone que existe
un campo `lineage` que consulta Neo4j, información que un atacante podría usar para
dirigir un ataque de denegación de servicio específicamente contra esa consulta, que
es más cara que el resto.

**¿Qué ataque frena el límite de profundidad?**
Consultas anidadas pensadas para que el servidor haga mucho trabajo por una sola
request del cliente (DoS por amplificación) — relevante acá porque `lineage` recorre
un grafo con un patrón de cualquier cantidad de saltos (`[:DERIVES|HAS_FEATURE|...*]`);
sin límite, nada impide construir una query que fuerce ese recorrido repetidas veces
en una sola llamada.

**¿Qué elemento de SAIF cubre este control?**
Autorización por campo (sobre `lineage`, no sobre todo el esquema) es *adaptar los
controles para mitigar amenazas propias de IA*: no todos los campos de este modelo
tienen el mismo costo ni el mismo nivel de sensibilidad, y el control refleja esa
diferencia real en vez de aplicar la misma regla a todo el esquema.